*This is the notebook for the training phase of this example*

## Feed Forward Network Demonstration (For MNIST)
**Goal:** Recognise the digit drawn on the image, shooting for 90%+ accuracy on Kaggle.

**Strategy:**
1. Architecture: 784 - 40 - 20 - 10
2. We do some basic preprocessing of the data, such as centering the image, and normalising the brightest pixel to 1.0
2. We have a data set of 42000 images, we will use the first 1000 images to quickly validate the accuracy of our program, and the remaining will be used to train.
3. We use Adam optimizer to optimize.
4. We save our training result to a `.cache` file.

This library works similarly to Pytorch, and I'm proud of that.

### 1. Import and Config

In [2]:
import math
import lktorch as lk
import os
import pandas as pd


CURRENT_DIR = os.getcwd()
WEIGHT_PATH = os.path.join(CURRENT_DIR, "training_data", "model_weight.cache")
DATA_PATH = os.path.join(CURRENT_DIR, "training_data", "train.csv")

### 2. Generate and preprocess

In [3]:
input = pd.read_csv(DATA_PATH)
row_count, column_count = input.shape

input_dataset = []
output_dataset = []
for i in range(row_count):
    input_row = input.loc[i].to_numpy()

    current_output = input_row[0]
    current_row = [0] * 10
    current_row[current_output] = 1
    output_dataset.append(current_row)
    input_dataset.append(input_row[1:])


def preprocess_row(current_row):
    result_row = [0] * 784
    moment_x = 0
    moment_y = 0
    sum = 0
    for i in range(0, 784):
        x = i // 28
        y = i % 28
        moment_x += x * current_row[i]
        moment_y += y * current_row[i]
        sum += current_row[i]
    centroid_x = moment_x // sum
    centroid_y = moment_y // sum

    centroid_x = 14 - centroid_x
    centroid_y = 14 - centroid_y

    for i in range(0, 784):
        x = i // 28
        y = i % 28

        x += centroid_x
        y += centroid_y

        if (x >= 0 and x < 28 and y >= 0 and y < 28):
            result_row[x * 28 + y] = current_row[i]

    ma = 0
    for i in result_row:
        ma = max(ma, i)
    for i in range(0, 784):
        result_row[i] = result_row[i] / ma

    return result_row

for i in range(len(input_dataset)):
    input_dataset[i] = preprocess_row(input_dataset[i])

print("Done fetching data!")

Done fetching data!


### 3. Prepare for the training

In [4]:

model = lk.Sequential([
    lk.LinearLayer(784, 40),
    lk.reLU_Layer(),
    lk.LinearLayer(40, 20),
    lk.reLU_Layer(),
    lk.LinearLayer(20, 10),
    lk.Sigmoid_Layer(),
    lk.SoftMax_Layer()
])


input_tensor = lk.Tensor(input_dataset)
output_tensor = lk.Tensor(output_dataset)

train_input_tensor = input_tensor.Slice([1000, 0], [41999, 783])
train_output_tensor = output_tensor.Slice([1000, 0], [41999, 9])

eval_input_tensor = input_tensor.Slice([0, 0], [999, 783])
eval_output_tensor = output_tensor.Slice([0, 0], [999, 9])


optimizer = lk.Adam(2, 0.9, 0.999)
optimizer.add_parameter(model.get_parameters())

def calculate_loss(model, input_data, output_data, LossFunction):
    tenso = model(input_data)
    tenso = LossFunction(tenso, output_data)
    return lk.Mean(tenso)

### 4. Training and saving to file

In [5]:

lr = 2
best = 1e18
for epoch in range(1, 101):
    BATCH = 42
    l = 0
    lim = train_input_tensor.dimension()[0]
    while (l < lim):
        optimizer.zero_gradient()

        r = min(l + BATCH, lim) - 1

        input_batch = train_input_tensor.Slice([l, 0], [r, 783])
        tenso = model(input_batch)
        tenso = lk.Mean(lk.MSELoss(tenso, train_output_tensor.Slice([l, 0], [r, 9])))

        tenso.backward()
        optimizer.step()
        l += BATCH
    print("epoch:", epoch)
    cur_loss = calculate_loss(model, eval_input_tensor, eval_output_tensor, lk.MSELoss).accessA([])
    print("loss: ", cur_loss)
    if (cur_loss < best):
        best = cur_loss
        lk.WriteFile(model.get_state_dict(), WEIGHT_PATH)
        print("File saved successfully!")


    lr *= 0.96
    optimizer.set_learning_rate(lr)


epoch: 1
loss:  0.07783760875463486
File saved successfully!
epoch: 2
loss:  0.046331822872161865
File saved successfully!
epoch: 3
loss:  0.027858933433890343
File saved successfully!
epoch: 4
loss:  0.02327696420252323
File saved successfully!
epoch: 5
loss:  0.020742500200867653
File saved successfully!
epoch: 6
loss:  0.02129541151225567
epoch: 7
loss:  0.018063774332404137
File saved successfully!
epoch: 8
loss:  0.017636477947235107
File saved successfully!
epoch: 9
loss:  0.016685327515006065
File saved successfully!
epoch: 10
loss:  0.0154644176363945
File saved successfully!
epoch: 11
loss:  0.01544610969722271
File saved successfully!
epoch: 12
loss:  0.014973320998251438
File saved successfully!
epoch: 13
loss:  0.014813032001256943
File saved successfully!
epoch: 14
loss:  0.014127104543149471
File saved successfully!
epoch: 15
loss:  0.01346198283135891
File saved successfully!
epoch: 16
loss:  0.012474803254008293
File saved successfully!
epoch: 17
loss:  0.01361146103590

KeyboardInterrupt: 

### 5. Verify model training result

In [7]:
def get_label(tensor):
    label = []
    sz = tensor.dimension()
    for i in range(sz[0]):
        cur = -1
        ma = -1
        for j in range(0, 10):
            current_w = tensor.accessA([i, j])
            if (current_w > ma):
                ma = current_w
                cur = j
        label.append(cur)
    return label

def calculate_fault(tensor, label):
    correct = 0
    my_label = get_label(tensor)
    for i in range(len(label)):
        if (my_label[i] == label[i]):
            correct += 1
    return correct / len(label)

label = []
for i in range(row_count):
    input_row = input.loc[i].to_numpy()
    label.append(input_row[0])


lk.deactivate_backprop()

model.load_state_dict(lk.ReadFile(WEIGHT_PATH))
tenso = model(input_tensor)
print("Accuracy: ", calculate_fault(tenso, label))


print("Sanity check: ")
print("Accuracy of output dataset: ", calculate_fault(output_tensor, label))

Accuracy:  0.9551428571428572
Sanity check: 
Accuracy of output dataset:  1.0
